In [1]:
#!/usr/bin/e4nv python3
"""
Análisis de error espacial: Compara la serie de tiempo del solitón sintético
con las series simuladas a lo largo de la línea media del canal (y = 1.5 m)
para identificar dónde se reproduce mejor la condición de borde impuesta.
"""
import numpy as np
import matplotlib.pyplot as plt
import json
import glob
import re
import os

plt.style.use('Solarize_Light2')
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=plt.cm.viridis(np.linspace(0, 1, 9)))

# ================== CARGAR CONFIGURACIÓN SIMULACIÓN ==================
carpeta_sim = "rund_yo"
with open(os.path.join(carpeta_sim, 'config.json'), 'r') as f:
    config = json.load(f)

nx = config['WIDTH']
ny = config['HEIGHT']
dx = config['dx']
dy = config['dy']

# Crear coordenadas
x = np.arange(0, nx) * dx
y = np.arange(0, ny) * dy

# Encontrar el índice j más cercano a y = 1.5 m
y_target = 1.5
j_target = np.argmin(np.abs(y - y_target))
y_actual = y[j_target]


# ================== CARGAR DATOS DE SIMULACIÓN ==================
def sort_files_numerically(files):
    return sorted(files, key=lambda x: int(re.search(r'(\d+)', x).group(1)))

def load_bin(filename, ny, nx):
    with open(filename, 'rb') as f:
        return np.fromfile(f, dtype=np.float32).reshape((ny, nx))


elev_files = sort_files_numerically(glob.glob(os.path.join(carpeta_sim, 'elev_*.bin')))
all_time_files = glob.glob(os.path.join(carpeta_sim, 'time_*.txt'))
time_files = [
    f for f in all_time_files
    if re.search(r'time_(\d+)\.txt$', os.path.basename(f))
]
time_files = time_files = sort_files_numerically(time_files)


# print(f"\nSimulación: {len(times_sim)} pasos temporales")
# print(f"Rango temporal simulación: [{times_sim[0]:.2f}, {times_sim[-1]:.2f}] s")

In [2]:
elev_indices = set([int(f.split("_")[-1].split(".")[0]) for f in elev_files])
time_indices = set([int(f.split("_")[-1].split(".")[0]) for f in time_files])
elev_indices = elev_indices.intersection(time_indices)
time_indices = time_indices.intersection(elev_indices)
assert len(elev_indices.symmetric_difference(time_indices))==0
elev_indices = np.array(list(time_indices))
time_indices = np.array(list(elev_indices))
assert len(elev_indices[:-1][np.diff(elev_indices)<0])==0
assert len(time_indices[:-1][np.diff(time_indices)<0])==0

In [3]:
!cat rund_yo/time_1.txt

0.0454336899611537


In [4]:
from pathlib import Path
from tqdm import tqdm

times_sim = np.array([
    float(Path(f"rund_yo/time_{i}.txt").read_text())
    for i in tqdm(time_indices)
])

100%|██████████| 18466/18466 [01:09<00:00, 263.80it/s] 


In [5]:
np.savetxt("./rund_yo/times_sim.txt", times_sim)

In [6]:
elev_files[0]

'rund_yo\\elev_1.bin'

In [9]:
from pathlib import Path

stack = np.empty((len(elev_indices), nx), dtype=np.float32)

base = Path("rund_yo")
for loop_index, index in tqdm(enumerate(elev_indices)):
    p = base / f"elev_{index}.bin"
    eta = load_bin(p, ny, nx)
    stack[loop_index,:] = eta[4,:]

18466it [04:59, 61.74it/s]


In [19]:
np.save("./rund_yo/stacked", stack)

In [20]:
!ls rund_yo/stacked.npy

rund_yo/stacked.npy
